In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis_scripts.graphs import set_size
from analysis_scripts.load_lammps import load_lammps

from pizza.dump import dump

plt.style.use('seaborn')
plt.style.use('tex')
dumpsdir = os.path.abspath("../dumps/")

In [ ]:
testname = "test1_prob1_rate1000_cutoff1.6"
testype = "parallel"

bonds_d = dump(os.path.join(dumpsdir, f"{testname}_bond_{testype}.lammpstrj"))
coord_d = dump(os.path.join(dumpsdir, f"{testname}_dump_{testype}.lammpstrj"))
try:
    coord_d.unwrap()
except Exception:
    print("Already unwrapped")


In [ ]:
bonds = bonds_d.all_atoms_info()[:, :, 1:]
bonds = np.concatenate(
    (np.repeat(np.full(bonds.shape[0], 1).cumsum(), bonds.shape[1]).reshape(
        bonds.shape[0], bonds.shape[1], 1
    ),
    bonds),
    axis=2,
)
itype = coord_d.all_atoms_info()[:, :, :5]
itype = np.concatenate(
    (np.repeat(np.full(itype.shape[0], 1).cumsum(), itype.shape[1]).reshape(
        itype.shape[0], itype.shape[1], 1
    ),
    itype),
    axis=2,
)


In [ ]:
itype2 = itype[(itype[:,:,2]==2)]
bonds2 = bonds[(bonds[:,:,3]==2)]
bonds3 = bonds[(bonds[:,:,3]==3)]

typedf = pd.DataFrame(itype2, columns=["time","id","type","x","y","z"])
bonds2df = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])
bonds3df = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])

if np.any(bonds2df[["time","type"]].groupby("time").count()>2):
    print("Detected multiple bond")
if np.any(bonds3df[["time","type"]].groupby("time").count()>2):
    print("Detected multiple bond")
if np.any(typedf[["time","type"]].groupby("time").count()>2):
    print("Detected multiple type assegnation")

In [ ]:
plt.figure(figsize=set_size(500))
ax = plt.gca()
ax2 = ax.twinx()
ax2.grid(False)

ax.set_xlabel("Time [It]")
ax.set_ylabel("Indexes")

ax.fill_between(bonds2[:,0],bonds2[:,2],bonds2[:,1], alpha =0.7)
ax.fill_between(bonds3[:,0],bonds3[:,2],bonds3[:,1], alpha =0.7)

ax.scatter(itype2[:,0],itype2[:,1], s=5)


plt.savefig(f"results/kymograph_{testname}_{testype}.pdf")